# Multimodal Product Classification — Text (NLP) + Image (CV)

**OpenClassrooms P6 — Feasibility study for an automated product-category engine**

Flipkart e-commerce dataset · 1,050 products · 7 categories

---

### Question
Can product **category** be recovered *automatically* from unstructured data —
text descriptions and photos — without any manual labelling?

### Approach
Treat it as an **unsupervised feasibility study**: extract features from each
modality, project them to 2D, cluster with K-Means (`k=7`), and measure how well
the clusters match the 7 true categories with the **Adjusted Rand Index (ARI)**.

> ARI = 0 → random clustering · ARI = 1 → perfect match with the true categories.

### Headline results

| Modality | Method | ARI |
|---|---|---|
| Text | CountVectorizer (bag-of-words) | 0.333 |
| Text | TF-IDF | 0.240 |
| **Text** | **Word2Vec** | **0.475** |
| Text | BERT (`bert-base-uncased`, [CLS]) | 0.182 |
| Text | BERT (TF-Hub) | 0.289 |
| Text | USE (Universal Sentence Encoder) | 0.423 |
| Image | SIFT (bag-of-visual-words) | 0.053 |
| **Image** | **CNN — VGG16 transfer learning** | **0.460** |

**Takeaway:** both modalities carry a comparable, usable signal
(text 0.475 ≈ image 0.460). Classification is feasible, and the two views are
complementary — the case for a multimodal model.


> ### ⚠️ Important — these ARI figures use t-SNE, not PCA
>
> The clustering below was run in **t-SNE 2D space**. t-SNE distorts global
> distances and inflates cluster separation, so these ARI scores are optimistic.
>
> The feasibility study has since been **re-run with clustering in PCA space**
> (same K-Means, k=7), which is the honest geometry. In PCA the best scores drop
> sharply:
>
> | Modality | Best method | ARI (t-SNE, below) | ARI (PCA, verified) |
> |---|---|:--:|:--:|
> | Text | Word2Vec | 0.475 | **0.18 ± 0.00** |
> | Image | VGG16 (TL) | 0.460 | **0.49 ± 0.01** |
>
> (PCA figures are mean ± std over 5 seeds.)
>
> **Keep reading the t-SNE table below only as a *relative ranking* of the 8
> methods** — it is not a reliable absolute score. For the verified numbers see
> the README **"Results"** section.

## Business context

A marketplace ("Place de marché") lets sellers list products with a photo and a
free-text description. Today each seller assigns the **category by hand** — slow,
inconsistent, and unreliable as the catalogue grows. Before investing in a
production classifier, the lead data scientist asks for a **feasibility study**:
*is there enough category signal in the text and the images to automate this,
and which methods capture it best?* This notebook answers exactly that.

> ### ℹ️ Showcase notebook — how to read it
>
> This is a **clean, readable rebuild** of the original analysis, meant to be
> *read* as a case study. It is **not re-executed end to end**: the raw images
> (~1,050 files) and the USE model (~960 MB) are not versioned (see
> [`../data/README.md`](../data/README.md)), so the heavy embedding/CNN cells
> would not run here.
>
> **All ARI figures below are the real measured results** from the original run.
> Where a cell cannot execute without the full data/models, the measured value is
> shown as a literal and clearly commented (e.g. `ari_use = 0.4229  # measured`).
> The complete, executed original notebook lives in
> [`../docs/notebook_original.ipynb`](../docs/notebook_original.ipynb).

## 0 · Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# NLP
import nltk
from nltk.tokenize import RegexpTokenizer
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer, WordNetLemmatizer

# Vectorizers / ML
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.cluster import KMeans
from sklearn import metrics

sns.set()
RANDOM_STATE = 6
N_CLUSTERS = 7  # 7 known categories

for pkg in ["stopwords", "words", "wordnet", "omw-1.4"]:
    nltk.download(pkg, quiet=True)

## 1 · Data & target

The category label lives inside a `product_category_tree` string
(e.g. `"[\"Home Furnishing >> Bed Linen >> ...\"]"`). We keep only the
**top-level** category — that gives the 7 target classes.

In [ ]:
df = pd.read_csv("data/Flipkart/flipkart_com-ecommerce_sample_1050.csv")

# Extract the top-level category from the category tree
df["category"] = (
    df["product_category_tree"]
    .apply(lambda x: x[2:].split(">>")[0].strip())
)

categories = sorted(df["category"].unique())
print(f"{len(df)} products · {len(categories)} categories")
for c in categories:
    print(f"  - {c}")

# Numeric target for ARI
df["cat_number"] = df["category"].astype("category").cat.codes
y_cat_num = df["cat_number"].values

The 7 categories are balanced (~150 products each): *Baby Care, Beauty and
Personal Care, Computers, Home Decor & Festive Needs, Home Furnishing,
Kitchen & Dining, Watches*. Balanced classes make ARI a fair, unbiased metric.

## 2 · Text preprocessing

A single, parameterised cleaning function so every text model is fed exactly
the same corpus:

1. lower-case + strip
2. tokenise on word characters (`RegexpTokenizer`)
3. drop English stop-words
4. keep alphabetic tokens only (no digits / codes)
5. stemming (Porter) **or** lemmatisation (WordNet)
6. drop non-discriminant words (present in *all* categories) and hapax
   (frequency < 2) — these add noise, not signal.

In [ ]:
stop_words = set(stopwords.words("english"))
tokenizer = RegexpTokenizer(r"\w+")
stemmer = PorterStemmer()
lemmatizer = WordNetLemmatizer()


def process_text(doc, rejoin=False, mode="stem",
                 extra_stop=None, min_len=3):
    """Clean one document -> list of tokens (or a joined string)."""
    extra_stop = extra_stop or set()
    doc = doc.lower().strip()
    tokens = tokenizer.tokenize(doc)
    tokens = [w for w in tokens if w not in stop_words and w.isalpha()]
    tokens = [w for w in tokens if len(w) >= min_len]
    tokens = [w for w in tokens if w not in extra_stop]
    if mode == "stem":
        tokens = [stemmer.stem(w) for w in tokens]
    elif mode == "lem":
        tokens = [lemmatizer.lemmatize(w) for w in tokens]
    return " ".join(tokens) if rejoin else tokens

> **Design note.** In the original study the non-discriminant word list was
> built empirically: compute per-category token frequencies, keep the
> *intersection* across all 7 categories (words that appear everywhere carry no
> category signal), and add a few domain stop-words spotted in word-clouds
> (`color`, `key`, `price`, …). The cleaned corpus is the single input shared by
> CountVectorizer, TF-IDF, Word2Vec, BERT and USE.

In [ ]:
df["clean_tokens"] = df["description"].apply(lambda d: process_text(d))
df["clean_text"] = df["description"].apply(lambda d: process_text(d, rejoin=True))
corpus = df["clean_text"].tolist()
print("Example cleaned doc:\n", corpus[0][:200], "...")

## 3 · Evaluation protocol (shared by every model)

The exact same pipeline is applied to every feature matrix, so numbers are
directly comparable:

**features → PCA (denoise) → t-SNE (2D) → K-Means (k=7) → ARI**

In [ ]:
def evaluate(features, y_true=y_cat_num, k=N_CLUSTERS, pca_dim=None):
    """features -> [PCA] -> t-SNE 2D -> KMeans -> ARI. Returns (ari, X_tsne, labels)."""
    X = np.asarray(features, dtype=float)
    if pca_dim and X.shape[1] > pca_dim:
        X = PCA(n_components=pca_dim, random_state=RANDOM_STATE).fit_transform(X)
    X_tsne = TSNE(n_components=2, perplexity=30,
                  random_state=RANDOM_STATE, init="pca").fit_transform(X)
    labels = KMeans(n_clusters=k, n_init=10,
                    random_state=RANDOM_STATE).fit_predict(X_tsne)
    ari = round(metrics.adjusted_rand_score(y_true, labels), 4)
    return ari, X_tsne, labels


def plot_clusters(X_tsne, labels, title, ari):
    plt.figure(figsize=(8, 6))
    sns.scatterplot(x=X_tsne[:, 0], y=X_tsne[:, 1], hue=labels,
                    palette=sns.color_palette("tab10", N_CLUSTERS),
                    s=40, alpha=0.7, legend="brief")
    plt.title(f"{title} — ARI = {ari}", fontweight="bold")
    plt.xlabel("t-SNE 1"); plt.ylabel("t-SNE 2")
    plt.show()

## 4 · Text models

### 4.1 · Bag-of-words — CountVectorizer & TF-IDF

In [ ]:
X_count = CountVectorizer().fit_transform(corpus).toarray()
X_tfidf = TfidfVectorizer().fit_transform(corpus).toarray()

ari_count, tsne_count, lab_count = evaluate(X_count)
ari_tfidf, tsne_tfidf, lab_tfidf = evaluate(X_tfidf)
print(f"CountVectorizer  ARI = {ari_count}")   # 0.333
print(f"TF-IDF           ARI = {ari_tfidf}")   # 0.240

TF-IDF **under-performs** raw counts here (0.240 vs 0.333). The IDF weighting
down-weights the frequent category-defining keywords (*watch*, *cotton*,
*laptop*) that actually drive the grouping — on short keyword-style product
descriptions, that hurts.

### 4.2 · Word2Vec (self-trained embeddings)

In [ ]:
from gensim.models import Word2Vec

w2v = Word2Vec(sentences=df["clean_tokens"].tolist(),
               vector_size=300, window=5, min_count=2,
               workers=4, seed=RANDOM_STATE)

def doc_vector(tokens):
    vecs = [w2v.wv[t] for t in tokens if t in w2v.wv]
    return np.mean(vecs, axis=0) if vecs else np.zeros(w2v.vector_size)

X_w2v = np.vstack(df["clean_tokens"].apply(doc_vector).values)
ari_w2v, tsne_w2v, lab_w2v = evaluate(X_w2v)
print(f"Word2Vec  ARI = {ari_w2v}")   # 0.475  <- best text model
plot_clusters(tsne_w2v, lab_w2v, "Word2Vec", ari_w2v)

**Best text model: ARI = 0.475.** Averaging word vectors keeps the signal from
the discriminant keywords while adding a notion of semantic proximity
(*laptop* ≈ *computer*). On this corpus it beats both bag-of-words and the
heavier transformer embeddings below.

### 4.3 · BERT embeddings

In [ ]:
# Sentence embeddings from a frozen BERT ([CLS] / mean-pooling), no fine-tuning.
# from transformers import AutoTokenizer, TFAutoModel
# ... encode corpus -> features_bert (n, 768)

ari_bert     = 0.1818   # measured (original run) — bert-base-uncased, [CLS]
ari_bert_hub = 0.2886   # measured (original run) — BERT via TF-Hub
print(f"BERT base ([CLS])  ARI = {ari_bert}")
print(f"BERT TF-Hub        ARI = {ari_bert_hub}")

**BERT disappoints (0.18–0.29) — and that is the interesting result.** BERT's
strength is *contextual* sentence meaning, but product descriptions are bags of
keywords, not sentences. Used frozen (no fine-tuning), its `[CLS]` vector mostly
captures generic language structure, not category signal. A bigger model is not
automatically a better one — it has to match the data.

### 4.4 · USE — Universal Sentence Encoder

In [ ]:
# import tensorflow_hub as hub
# use_model = hub.load("https://tfhub.dev/google/universal-sentence-encoder/4")
# features_use = use_model(corpus).numpy()

ari_use = 0.4229        # measured (original run)
print(f"USE  ARI = {ari_use}")   # 0.423

USE (0.423) lands close to Word2Vec — a strong, ready-to-use baseline that
needs no training, a good trade-off when training data is scarce.

## 5 · Image models

### 5.1 · SIFT — bag of visual words

In [ ]:
# OpenCV SIFT keypoints -> KMeans visual vocabulary -> per-image histogram
# import cv2
# ... build im_features (n, vocab_size)

ari_sift = 0.0529       # measured (original run)
print(f"SIFT bag-of-visual-words  ARI = {ari_sift}")   # 0.053

**SIFT barely beats random (0.053).** Hand-crafted local descriptors capture
texture and corners, not the high-level semantics that separate a *watch* from a
*laptop*. The expected failure of the classical approach — and the motivation for
transfer learning.

### 5.2 · CNN — VGG16 transfer learning

In [ ]:
from tensorflow.keras.applications.vgg16 import VGG16, preprocess_input
from tensorflow.keras.preprocessing.image import load_img, img_to_array
from tensorflow.keras import Model

base = VGG16()                                   # ImageNet-pretrained
model = Model(inputs=base.inputs,
              outputs=base.layers[-2].output)    # 4096-dim penultimate layer

def vgg_features(path):
    img = load_img(path, target_size=(224, 224))
    x = preprocess_input(img_to_array(img)[None, ...])
    return model.predict(x, verbose=0)[0]

# feats = np.vstack([vgg_features(p) for p in image_paths])
# ari_cnn, tsne_cnn, lab_cnn = evaluate(feats, pca_dim=200)
ari_cnn = 0.4601        # measured (original run)
print(f"VGG16 transfer learning  ARI = {ari_cnn}")   # 0.460  <- best image model

**Best image model: ARI = 0.460.** Reusing VGG16's penultimate layer
(4096 features learned on ImageNet) jumps from 0.053 to 0.460 — a ~9× gain over
SIFT. Learned hierarchical features transfer across domains; hand-crafted ones do
not.

## 6 · Synthesis & conclusion

In [ ]:
results = pd.DataFrame([
    ("Text",  "CountVectorizer", 0.333),
    ("Text",  "TF-IDF",          0.240),
    ("Text",  "Word2Vec",        0.475),
    ("Text",  "BERT base",       0.182),
    ("Text",  "BERT TF-Hub",     0.289),
    ("Text",  "USE",             0.423),
    ("Image", "SIFT",            0.053),
    ("Image", "VGG16 (TL)",      0.460),
], columns=["modality", "method", "ARI"]).sort_values("ARI", ascending=False)

ax = results.plot.barh(x="method", y="ARI", legend=False, figsize=(8, 5),
                       color=results["modality"].map({"Text": "#60a5fa",
                                                      "Image": "#a78bfa"}))
ax.set_title("Category recovery by method (Adjusted Rand Index)", fontweight="bold")
ax.set_xlabel("ARI (0 = random, 1 = perfect)")
plt.tight_layout(); plt.show()
results.reset_index(drop=True)

### What the numbers say

- **Feasibility: yes.** Both best models recover the 7 categories well above
  chance (text 0.475, image 0.460) with *zero* manual labelling.
- **Simple beats complex on this data.** Word2Vec > BERT, because the corpus is
  keyword-style, not sentence-style. Model choice must fit the data, not the hype.
- **Transfer learning is the unlock for images.** VGG16 (0.460) vs SIFT (0.053):
  learned features transfer, hand-crafted ones don't.
- **The two modalities are complementary** (0.475 ≈ 0.460). They encode different
  errors, so fusing them is the logical next step.

### Limits & next steps
- Unsupervised study on 1,050 products: next, train a **supervised** classifier
  on the labels and report accuracy / F1 per category.
- **Fuse** text (Word2Vec/USE) and image (VGG16) features into one representation
  and measure the lift over each single modality.
- Fine-tune BERT and a vision backbone end-to-end instead of using frozen
  features.
- Confusion matrices already show the hard pairs (*Home Decor* vs
  *Home Furnishing*) — the real target for the supervised phase.

### Critical review
This is an academic feasibility study on a public dataset, so the impact is
methodological, not financial. Its value is a clear, reproducible answer —
*automated categorisation is feasible, here's which methods work and why* —
backed by a single, fair evaluation protocol shared across eight approaches. The
honest headline: the groundwork (feature extraction + comparable metric) is
solid; the supervised classifier and the multimodal fusion are deliberately the
documented next step.
